# 06 — Inference

Use `prices.predict.predict` — the same code path the Flask app calls — to score new rows with the tuned Ridge model.

We temporarily point `config.ARTIFACTS_DIR` at `notebooks/_artifacts/` so we don't depend on a full repo-level training run.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SCRATCH = REPO_ROOT / "notebooks" / "_artifacts"

import pandas as pd

import config
from prices import predict as predict_mod

X_test_raw = pd.read_parquet(SCRATCH / "X_test_raw.parquet")
sample = X_test_raw.head(5).reset_index(drop=True)
sample

,year,age,beds,baths,home_size,parcel_size,pool,dist_cbd,dist_lakes,x_coord,y_coord
0,2004,23,3,2.0,1832,13200,0,24090.26,896.92,481931.2,1589851
1,2003,0,4,2.0,2010,8378,1,21860.87,856.49,603543.4,1551759
2,2003,39,3,2.0,1777,14250,0,6510.52,98.00,535733.6,1552446
3,2001,8,4,2.0,1934,13754,1,11245.10,414.98,497935.1,1531682
4,2002,19,4,3.0,1827,7498,1,17120.15,6205.15,540009.6,1475149


## Predict via `prices.predict`

In [2]:
original_dir = config.ARTIFACTS_DIR
config.ARTIFACTS_DIR = SCRATCH
try:
    predictions = predict_mod.predict("ridge", sample)
finally:
    config.ARTIFACTS_DIR = original_dir

out = sample.assign(predicted_price=predictions.values)
out

,year,age,beds,baths,home_size,parcel_size,pool,dist_cbd,dist_lakes,x_coord,y_coord,predicted_price
0,2004,23,3,2.0,1832,13200,0,24090.26,896.92,481931.2,1589851,152411.180988
1,2003,0,4,2.0,2010,8378,1,21860.87,856.49,603543.4,1551759,155040.179676
2,2003,39,3,2.0,1777,14250,0,6510.52,98.00,535733.6,1552446,198391.138500
3,2001,8,4,2.0,1934,13754,1,11245.10,414.98,497935.1,1531682,170337.402371
4,2002,19,4,3.0,1827,7498,1,17120.15,6205.15,540009.6,1475149,156344.229663


## Single-row example

Build a one-row DataFrame from scratch — the same shape the Flask `POST /api/predict` endpoint accepts.

In [3]:
single = pd.DataFrame([{
    "year": 2010, "age": 5, "beds": 3, "baths": 2,
    "home_size": 1800, "parcel_size": 6000, "pool": 0,
    "dist_cbd": 8000, "dist_lakes": 4000,
    "x_coord": 540000, "y_coord": 1500000,
}])

config.ARTIFACTS_DIR = SCRATCH
try:
    pred = predict_mod.predict("ridge", single)
finally:
    config.ARTIFACTS_DIR = original_dir

print(f"predicted price: ${pred.iloc[0]:,.0f}")

predicted price: $263,879
